# MPS topology and Dmax convergence

Compare routed CPU MPS with an exact 10-qubit long-range circuit and inspect automated convergence evidence.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
rng = np.random.default_rng(41)
circuit = QuantumCircuit(10)
for layer in range(3):
    for wire in range(10):
        circuit.ry(float(rng.uniform(-1, 1)), wire)
    order = rng.permutation(10)
    for index in range(0, 10, 2):
        circuit.rzz(float(rng.uniform(-0.8, 0.8)), int(order[index]), int(order[index + 1]))

reference, reference_ms, _ = benchmark(lambda: np.asarray(Statevector.from_instruction(circuit).data))
backend = MettleQBackend(
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
    mps_routing_strategy="lookahead",
)
compiled = transpile(circuit, backend, optimization_level=1)

def run_mps():
    return np.asarray(backend.run(compiled, shots=1, return_statevector=True, execution_report=True).result().data(0)["statevector"])

candidate, mettleq_ms, _ = benchmark(run_mps)
error = phase_aligned_statevector_error(reference, candidate)
diagnostics = backend.last_mps_diagnostics[-1]
accuracy = backend.last_mps_accuracy_reports[-1]
convergence_estimator = MettleQEstimatorV2(
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
    mps_convergence_bond_dimensions=(8, 16, 32),
    mps_convergence_atol=5e-5,
)
convergence_result = convergence_estimator.run([(circuit, SparsePauliOp("IIIIIIIIIZ"))]).result()[0]
convergence = convergence_result.metadata["mettleq_mps_convergence"]
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/14_mps_topology_and_convergence.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-aligned MPS state atol=8e-5 and convergence report",
    passed=error <= 8e-5 and convergence["converged"] and accuracy["passed"],
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": error, "peak_bond": diagnostics["maximum_bond_dimension_reached"], "accuracy": accuracy, "convergence": convergence},
)